<a href="https://colab.research.google.com/github/CreatorPoints/PhotonCoreV2/blob/main/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Super Wav2Lip HD (Python 3.12 Patch)

Rebuilt pipeline with runtime legacy attribute injection for Python 3.12 / NumPy 2.x compatibility.

In [13]:
import sys, os, subprocess

print("📦 Locking wheel-compatible dependencies...")
try:
    # Force reinstall numpy and scipy to address 'numpy.dtype size changed' binary incompatibility
    result = subprocess.run([
        sys.executable, "-m", "pip", "install", "--force-reinstall",
        "numpy==1.26.4", "scipy==1.12.0", "librosa==0.9.2", "numba>=0.59.0"
    ], check=True, capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
except subprocess.CalledProcessError as e:
    print("Error during pip install:")
    print("STDOUT:", e.stdout)
    print("STDERR:", e.stderr)
    raise # Re-raise the exception after printing details

print("✅ Dependencies locked successfully!")

📦 Locking wheel-compatible dependencies...
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached scipy-1.12.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
  Using cached librosa-0.9.2-py3-none-any.whl.metadata (8.2 kB)
  Using cached resampy-0.4.3-py3-none-any.whl.metadata (3.0 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 3.9 MB/s eta 0:00:00
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
Using cached scipy-1.12.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.8 MB)
Using cached librosa-0.9.2-py3-none-any.whl (214 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 110.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 7

## 1. Installation & Model Setup

In [14]:
import os
%cd /content

if not os.path.exists('/content/wav2lip-HD'):
    !git clone https://github.com/indianajson/wav2lip-HD.git

basePath = "/content/wav2lip-HD"
%cd {basePath}

wav2lipFolderName = 'Wav2Lip-master'
gfpganFolderName = 'GFPGAN-master'
wav2lipPath = basePath + '/' + wav2lipFolderName
gfpganPath = basePath + '/' + gfpganFolderName

!mkdir -p {wav2lipPath}/face_detection/detection/sfd/
!mkdir -p {wav2lipPath}/checkpoints/
!mkdir -p {basePath}/inputs
!mkdir -p {basePath}/outputs

# Download detector & models from HuggingFace mirrors
!wget -q 'https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth' -O {wav2lipPath}/face_detection/detection/sfd/s3fd.pth
!wget -q 'https://huggingface.co/Nekochu/Wav2Lip/resolve/main/wav2lip_gan.pth' -O {wav2lipPath}/checkpoints/wav2lip_gan.pth
!wget -q 'https://huggingface.co/Nekochu/Wav2Lip/resolve/main/wav2lip.pth' -O {wav2lipPath}/checkpoints/wav2lip.pth

# Setup GFPGAN for upscaling
!cd {gfpganPath} && python setup.py develop > /dev/null 2>&1
!wget -q https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth -P {gfpganPath}/experiments/pretrained_models

from IPython.display import clear_output
clear_output()
print("✅ Setup Complete! Upload your audio (bruh.mp3) and video (guy.mp4) into /content/wav2lip-HD/inputs/")

✅ Setup Complete! Upload your audio (bruh.mp3) and video (guy.mp4) into /content/wav2lip-HD/inputs/


## 2. Synchronize Video and Speech

In [16]:
import os, sys, builtins
import numpy as np
from types import ModuleType

# Inject legacy NumPy aliases directly in runtime memory
np.complex = complex
np.float = float
np.int = int
np.bool = bool
builtins.float32 = np.float32
builtins.complex128 = np.complex128

basePath = "/content/wav2lip-HD"
wav2lipFolderName = "/content/wav2lip-HD/Wav2Lip-master"

inputAudio = 'bruh.mp3' #@param{type:"string"}
inputVideo = 'guy.mp4' #@param{type:"string"}
model = "wav2lip" #@param ["wav2lip", "wav2lip_gan"] {type:"string"}

inputAudioPath = basePath + '/inputs/' + inputAudio
inputVideoPath = basePath + '/inputs/' + inputVideo
lipSyncedOutputPath = basePath + '/outputs/result.mp4'

# --- Start of workaround for librosa ImportError ---
# The project expects librosa==0.7.0, but 0.9.2 was installed, which removed 'deprecate_positional_args'.
# This patch temporarily adds a dummy function to allow imports to succeed.
def dummy_deprecate_positional_args(f):
    return f

# Ensure librosa.util.decorators is loaded and patch it if 'deprecate_positional_args' is missing
if 'librosa.util.decorators' not in sys.modules:
    try:
        import librosa.util.decorators as _librosa_decorators
        sys.modules['librosa.util.decorators'] = _librosa_decorators
    except ImportError:
        # Fallback to creating a mock module if the import itself fails (unlikely given traceback)
        _librosa_decorators = ModuleType('librosa.util.decorators')
        sys.modules['librosa.util.decorators'] = _librosa_decorators
        # Add 'deprecated' as well, as it's imported alongside in librosa.filters
        setattr(_librosa_decorators, 'deprecated', dummy_deprecate_positional_args)
        print("Created mock librosa.util.decorators module due to import error.")

if not hasattr(sys.modules['librosa.util.decorators'], 'deprecate_positional_args'):
    setattr(sys.modules['librosa.util.decorators'], 'deprecate_positional_args', dummy_deprecate_positional_args)
    print("Patched librosa.util.decorators with dummy 'deprecate_positional_args'.")
# --- End of workaround ---

# Run Wav2Lip directly inside python runtime
os.chdir(wav2lipFolderName)
sys.argv = [
    'inference.py',
    '--checkpoint_path', f'checkpoints/{model}.pth',
    '--face', inputVideoPath,
    '--audio', inputAudioPath,
    '--outfile', lipSyncedOutputPath
]

print("🚀 Synthesizing lip-sync...")
exec(open('inference.py').read())
print("✅ Lip-sync complete! Output saved to /content/wav2lip-HD/outputs/result.mp4")

🚀 Synthesizing lip-sync...


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## 3. Boost Resolution with GFPGAN

In [ ]:
import cv2, os, subprocess
from tqdm import tqdm
from os import path

basePath = "/content/wav2lip-HD"
outputPath = basePath + '/outputs'
inputVideoPath = outputPath + '/result.mp4'
unProcessedFramesFolderPath = outputPath + '/frames'

if not os.path.exists(unProcessedFramesFolderPath):
    os.makedirs(unProcessedFramesFolderPath)

vidcap = cv2.VideoCapture(inputVideoPath)
numberOfFrames = int(vidcap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = vidcap.get(cv2.CAP_PROP_FPS)
print(f"FPS: {fps} | Frames: {numberOfFrames}")

for frameNumber in tqdm(range(numberOfFrames)):
    _, image = vidcap.read()
    if image is not None:
        cv2.imwrite(path.join(unProcessedFramesFolderPath, str(frameNumber).zfill(4) + '.jpg'), image)

gfpganFolderName = basePath + '/GFPGAN-master'
!cd {gfpganFolderName} && python inference_gfpgan.py -i {unProcessedFramesFolderPath} -o {outputPath} -v 1.3 -s 2 --only_center_face --bg_upsampler None

restoredFramesPath = outputPath + '/restored_imgs/'
dir_list = sorted(os.listdir(restoredFramesPath))

img_array = []
for filename in tqdm(dir_list):
    img = cv2.imread(restoredFramesPath + filename)
    if img is not None:
        height, width, _ = img.shape
        img_array.append(img)

temp_silent_video = outputPath + '/temp_upscaled.mp4'
out = cv2.VideoWriter(temp_silent_video, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
for img in img_array:
    out.write(img)
out.release()

# Re-attach audio track cleanly via FFmpeg
final_output = outputPath + '/final_hd_output.mp4'
audio_source = basePath + '/inputs/bruh.mp3'
subprocess.run(f"ffmpeg -y -i {temp_silent_video} -i {audio_source} -c:v copy -c:a aac -shortest {final_output}", shell=True)

from IPython.display import clear_output
clear_output()
print(f"🎉 HD Upscaling Complete! Download your final video from: {final_output}")

## 4. Clear Cached Files

In [ ]:
%cd /content/wav2lip-HD/
!rm -rf inputs/* outputs/*
!mkdir -p inputs outputs

from IPython.display import clear_output
clear_output()
print("🧹 Workspace cleared and ready for the next clip!")